In [ ]:
import io
import chess.pgn
import pandas as pd
import numpy as np
import requests
import zstandard 
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone, BaseEstimator, TransformerMixin
from stockfish import Stockfish

df = pd.read_csv('games_2017-08.csv')

In [2]:
stockfish = Stockfish(path=r"C:\Users\yazid\Downloads\stockfish-windows-x86-64-universal\stockfish\stockfish-windows-x86-64-universal.exe", depth=12, parameters={"Threads": 2})

In [17]:
def evaluate_position(stockfish, board):
    if board.is_checkmate():
        return -10000 if board.turn == chess.WHITE else 10000
    if board.is_game_over():
        return 0
    stockfish.set_fen_position(board.fen())
    result = stockfish.get_evaluation()
    if result['type'] == 'cp':
        value = result['value']
    else:
        value = 10000 if result['value'] > 0 else -10000
    return value if board.turn == chess.WHITE else -value

In [3]:
RANDOM_STATE = 42
CV = 5

In [4]:
PIECE_VALUE = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9
}

def material(board, color):
    return sum(PIECE_VALUE.get(p.piece_type, 0)
        for p in board.piece_map().values() if p.color == color)

def extract_features(stockfish, game, n_moves=10):
    h = game.headers
    base, inc = h['TimeControl'].split('+')
    board = game.board()
    clocks = {chess.WHITE: float('nan'), chess.BLACK: float('nan')}
    plies = 0
    last_node = None
    eval_cp = float('nan')
    for node in game.mainline():
        mover = board.turn
        board.push(node.move)
        c = node.clock()
        if c is not None:
            clocks[mover] = c
        plies += 1
        last_node = node
        if plies == 2 * n_moves:
            eval_cp = evaluate_position(stockfish, board)
            break
    if plies < 2 * n_moves:
        return None
    if last_node.is_end():
        return None
    if board.is_game_over():
        return None
    w_elo, b_elo = int(h['WhiteElo']), int(h['BlackElo'])
    return {
        'game_id': h['Site'].split('/')[-1],
        'date': h['UTCDate'], 'time': h['UTCTime'],
        'white': h['White'], 'black': h['Black'], 
        'white_elo': w_elo, 'black_elo': b_elo,
        'elo_diff': w_elo - b_elo,
        'base_sec': int(base), 'inc_sec': int(inc),
        'material_diff': material(board, chess.WHITE) - material(board, chess.BLACK), 'white_clock': clocks[chess.WHITE],
        'black_clock': clocks[chess.BLACK], 'clock_diff': clocks[chess.WHITE] - clocks[chess.BLACK],
        'fen': board.fen(),
        'eval_white': eval_cp,
        'result': h['Result']
    }

def read_games(stockfish, url, max_games=100, n_moves=10):
    resp = requests.get(url, stream=True)
    rows = []
    stream = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(resp.raw), encoding='utf-8')
    while len(rows) < max_games:
        game = chess.pgn.read_game(stream)
        if game is None:
            break
        try:
            features = extract_features(stockfish, game, n_moves)
            if features is not None:
                rows.append(features)
        except (KeyError, ValueError):
            continue
    resp.close()
    return pd.DataFrame(rows)
        

In [5]:
df = df[df['result'].isin(['1-0', '0-1'])].copy()
df['white_won'] = (df['result'] == '1-0').astype(int)

In [6]:
df.to_csv('games_2017-08.csv', index=False)
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

In [7]:
cut = int(len(df) * 0.8)
df_train = df.iloc[:cut].copy()
df_test = df.iloc[cut:].copy()

In [8]:
X_train = df_train.drop(columns=['game_id', 'result', 'white_won', 'date', 'time', 'white', 'black', 'fen'])
y_train = df_train['white_won']
X_test = df_test.drop(columns=['game_id', 'result', 'white_won', 'date', 'time', 'white', 'black', 'fen'])
test_id = df_test['game_id']

In [9]:
COLUMN_TYPE = {'white_elo': 'num', 'black_elo': 'num', 'elo_diff': 'num', 'base_sec': 'num', 'inc_sec': 'num', 'material_diff': 'num',
               'white_clock': 'num', 'black_clock': 'num', 'clock_diff': 'num', 'eval_white': 'num'}


In [10]:
def split_by_type(columns):
    result = {'num': []}
    for col in columns:
        t = COLUMN_TYPE[col]
        result[t].append(col)
    return result

FEATURE_GROUPS = [
    ('core', ['white_elo', 'black_elo', 'elo_diff']), 
    ( '+clock', ['base_sec', 'inc_sec', 'white_clock', 'black_clock', 'clock_diff']),
    ( '+material', ['material_diff']),
    ('+position', ['eval_white'])
]

EXPERIMENT_COLUMNS = ['white_elo', 'black_elo', 'elo_diff', 'base_sec', 'inc_sec', 'white_clock', 'black_clock', 'clock_diff', 'material_diff', 'eval_white']
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [19]:
models =  {
    'xgboost': XGBClassifier(random_state=RANDOM_STATE),
    'logistic': LogisticRegression(max_iter=10000),
    'catboost': CatBoostClassifier(random_state=RANDOM_STATE, verbose=False, allow_writing_files=None), 
    'random_forest': RandomForestClassifier(random_state=RANDOM_STATE)
}

param_distributions = {
    'xgboost': {'model__max_depth': [2,3,4,5], 'model__learning_rate': [0.01, 0.03, 0.1], 'model__n_estimators': [100,300,600], 'model__subsample': [0.7, 0.85, 1.0], 'model__colsample_bytree': [0.6, 0.8, 1.0], 'model__min_child_weight': [1,3,5]},
    'logistic': {'model__C': [0.01, 0.1, 1, 10]},
    'catboost': {'model__max_depth': [3,4,5,6], 'model__learning_rate': [0.03, 0.05, 0.1], 'model__iterations': [300,500,1000], 'model__l2_leaf_reg': [1,3,5,10]},
    'random_forest': {'model__max_depth': [3,4,5,6,8, None], 'model__min_samples_leaf': [1,2,4,8,12], 'model__min_samples_split': [2,5,10,20], 'model__max_features': ['sqrt', 'log2', 0.5], 'model__max_samples': [0.6,0.8,None]}
}

family = {
    'xgboost': 'trees',
    'logistic': 'linear',
    'catboost': 'trees', 
    'random_forest': 'trees'
}


In [12]:
def get_dummy_prediction():
    dummy = DummyClassifier(strategy='constant', constant=1, random_state=RANDOM_STATE)
    score = -cross_val_score(estimator=dummy, X=X_train, y=y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return score

def build_preprocessor(num_cols):
    num_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    return ColumnTransformer(transformers=[('num', num_pipeline, num_cols)])

def build_pipeline(preprocessor, model):
    return Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

def evaluate_model(EXPERIMENT_COLUMNS, model_name, model):
    parts = split_by_type(EXPERIMENT_COLUMNS)
    preprocessor = build_preprocessor(parts['num'])
    pipeline = build_pipeline(preprocessor, model)
    score = cross_val_score(estimator=pipeline, X=X_train, y=y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return {'model': model_name, 'score': score.mean()}

def tune_model(COLUMNS, model, model_name, param_distributions, n_iter=30):
    parts = split_by_type(COLUMNS)
    preprocessor = build_preprocessor(parts['num'])
    pipeline = build_pipeline(preprocessor, model)
    search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_distributions, n_iter=n_iter, n_jobs=-1, scoring='accuracy', return_train_score=True, refit=False, random_state=RANDOM_STATE, error_score='raise')
    search.fit(X_train, y_train)
    results = pd.DataFrame(search.cv_results_)
    results['model'] = model_name
    results['train_accuracy'] = results['mean_train_score']
    results['validation_accuracy'] = results['mean_test_score']
    best_result = select_best_result(results)
    return search, results, best_result

def select_best_result(results):
    if results.empty:
        raise ValueError('result is empty')
    else:
        return results.loc[results['validation_accuracy'].idxmax()]

def score_columns(chosen, model):
    parts = split_by_type(chosen)
    preprocessor = build_preprocessor(num_cols=parts['num'])
    pipeline = build_pipeline(preprocessor=preprocessor, model=model)
    score = cross_val_score(estimator=pipeline, X=X_train, y=y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return score.mean()

In [ ]:
model_evaluation = []
for model_name, model in models.items():
    model_evaluation.append(evaluate_model(EXPERIMENT_COLUMNS, model_name, model))


In [32]:
model_evaluation = pd.DataFrame(model_evaluation)
model_evaluation.sort_values(by='score', ascending=False)


,model,score
2,catboost,0.655371
1,logistic,0.652761
3,random_forest,0.647552
0,xgboost,0.612096


In [ ]:
rows, used = [], []

for name, cols in FEATURE_GROUPS:
    used = used + cols
    row = {'step': name, 'n_features': len(used)}
    for model_name, model in models.items():
        row[model_name] = score_columns(used, model)
    rows.append(row)
ablation_results = pd.DataFrame(rows)


In [41]:
ablation_results

,step,n_features,xgboost,logistic,catboost,random_forest
0,core,3,0.581851,0.616525,0.614699,0.576381
1,+clock,8,0.588370,0.618609,0.607663,0.599843
2,+material,9,0.584460,0.634512,0.624345,0.607145
3,+position,10,0.612096,0.652761,0.655371,0.647552


In [20]:

FINAL_COLUMNS = ['white_elo', 'black_elo', 'elo_diff', 'base_sec', 'inc_sec', 'material_diff', 'white_clock', 'black_clock', 'clock_diff', 'eval_white']
searches = {}
tuning_results = []
best_results = []
for model_name, model in models.items():
    search, results, best_result = tune_model(FINAL_COLUMNS, model, model_name, param_distributions[model_name], 30)
    searches[model_name] = search
    tuning_results.append(results)
    best_results.append(best_result)
all_tuning_results = pd.concat(tuning_results, ignore_index=True)
best_results = pd.DataFrame(best_results).sort_values(by='validation_accuracy', ascending=False)

c:\Users\yazid\anaconda3\envs\ds_env\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 4 is smaller than n_iter=30. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [21]:
best_results['family'] = best_results['model'].map(family)
best_models_by_family = best_results.loc[best_results.groupby('family')['validation_accuracy'].idxmax()]

In [25]:
best_models_by_family['model']

3          logistic
11    random_forest
Name: model, dtype: str

In [28]:
def make_submission(best_models_by_family):
    predictions_proba = []
    parts = split_by_type(FINAL_COLUMNS)
    for _, results in best_models_by_family.iterrows():
        model = clone(models[results['model']])
        preprocessor = build_preprocessor(parts['num'])
        pipeline = build_pipeline(preprocessor, model)
        pipeline.set_params(**results['params'])
        pipeline.fit(X_train, y_train)
        prediction_proba = pipeline.predict_proba(X_test)[:,1]
        predictions_proba.append(prediction_proba)
    avg_proba = np.mean(predictions_proba, axis=0)
    ensemble_prediction = (avg_proba > 0.5).astype(int)
    submission = pd.DataFrame({'game_id': test_id, 'white_won': ensemble_prediction})
    submission.to_csv('./data/submission.csv', index=False)

In [29]:
make_submission(best_models_by_family)